In [1]:
%run ../../../../imports_common.py -e "loading-langchain"

In [ ]:
from   pathlib import Path
from   velari_core.core import read_root_dir
from   velari_ai.integrations.langchain.loaders.corpus import DocumentCorpus
from   velari_ai.integrations.langchain.loaders.loader import WebBaseLoader, LocalFileLoader

from   IPython.display import display

## Web Content

In [3]:
urls = [
    "https://docs.langchain.com/oss/python/integrations/document_loaders",
    "https://docs.langchain.com/oss/python/langchain/agents",
    "https://lilianweng.github.io/posts/2023-06-23-agent/",
    "https://lilianweng.github.io/posts/2023-03-15-prompt-engineering/",
    "https://lilianweng.github.io/posts/2023-10-25-adv-attack-llm/",
]

In [4]:
corpus    = DocumentCorpus(WebBaseLoader(urls)).load()   # step 1: core metadata (source, timestamp, content)
docs      = corpus.docs
doc_sz    = len(docs)
core_keys = set(docs[0].metadata)

display(doc_sz)
sorted(core_keys)                                         # before enrich(): no page.* keys yet

5

['content.metrics.length',
 'content.preview',
 'source.info.relative',
 'source.info.uri',
 'source.properties.is_remote',
 'timestamp.acquired_at']

In [5]:
corpus.enrich()                                          # step 2: further metadata, added in place to the same documents

display(docs[0].metadata["page.title"])
sorted(set(docs[0].metadata) - core_keys)                # after enrich(): the keys it added (page.*)

'Document loader integrations - Docs by LangChain'

['page.description',
 'page.description_length',
 'page.language',
 'page.links',
 'page.title']

In [6]:
corpus.df_metadata.head(2)

,meta.source.info.uri,meta.source.info.relative,meta.source.properties.is_remote,meta.timestamp.acquired_at,meta.content.preview,meta.content.metrics.length,meta.page.title,meta.page.description,meta.page.description_length,meta.page.language,meta.page.topic,meta.content.metrics.char_sz,meta.content.metrics.word_sz,meta.content.metrics.token_sz,meta.content.metrics.page_sz
0,https://docs.langchain.com/oss/python/integrat...,oss/python/integrations/document_loaders,True,2026-10-03T03:31:15.449478+00:00,Document loader integrations - Docs by LangCha...,7856,Document loader integrations - Docs by LangChain,Integrate with document loaders using LangChai...,55,en,NaN,7856,1046,1519,4.557
1,https://docs.langchain.com/oss/python/langchai...,oss/python/langchain/agents,True,2026-10-03T03:31:15.622123+00:00,Agents - Docs by LangChain Documentation Index...,52374,Agents - Docs by LangChain,Build a LangChain agent: a model calling tools...,136,en,NaN,52374,8308,12221,36.663


In [7]:
corpus.df_content[['page_content', 'meta.content.preview', 'meta.content.metrics.length']]

,page_content,meta.content.preview,meta.content.metrics.length
0,Document loader integrations - Docs by LangCha...,Document loader integrations - Docs by LangCha...,7856
1,Agents - Docs by LangChain Documentation Index...,Agents - Docs by LangChain Documentation Index...,52374
2,LLM Powered Autonomous Agents | Lil'Log Lil'Lo...,LLM Powered Autonomous Agents | Lil'Log Lil'Lo...,43559
3,Prompt Engineering | Lil'Log Lil'Log | Posts A...,Prompt Engineering | Lil'Log Lil'Log | Posts A...,29847
4,Adversarial Attacks on LLMs | Lil'Log Lil'Log ...,Adversarial Attacks on LLMs | Lil'Log Lil'Log ...,49434


In [8]:
corpus.df_docs.filter(like="metrics")

,meta.content.metrics.length,meta.content.metrics.char_sz,meta.content.metrics.word_sz,meta.content.metrics.token_sz,meta.content.metrics.page_sz
0,7856,7856,1046,1519,4.557
1,52374,52374,8308,12221,36.663
2,43559,43559,6640,9443,28.329
3,29847,29847,4616,6954,20.862
4,49434,49434,7161,12174,36.522


In [9]:
corpus.df_metrics_stats

,meta.content.metrics.char_sz,meta.content.metrics.word_sz,meta.content.metrics.token_sz,meta.content.metrics.page_sz
sum,183070.000,27771.000,42311.000,126.933
min,7856.000,1046.000,1519.000,4.557
max,52374.000,8308.000,12221.000,36.663
std,18260.462,2852.555,4453.950,13.362
mean,36614.000,5554.200,8462.200,25.387
median,43559.000,6640.000,9443.000,28.329
skew,-0.814,-0.796,-0.737,-0.737
kurtosis,-0.816,-0.737,-0.840,-0.840


In [10]:
corpus.df_docs.head(1).T

,0
page_content,Document loader integrations - Docs by LangCha...
meta.source.info.uri,https://docs.langchain.com/oss/python/integrat...
meta.source.info.relative,oss/python/integrations/document_loaders
meta.source.properties.is_remote,True
meta.timestamp.acquired_at,2026-10-03T03:31:15.449478+00:00
meta.content.preview,Document loader integrations - Docs by LangCha...
meta.content.metrics.length,7856
meta.page.title,Document loader integrations - Docs by LangChain
meta.page.description,Integrate with document loaders using LangChai...
meta.page.description_length,55


## LocalFile

In [11]:
faq_path   = Path(read_root_dir()) / "data/cs/datasets/faq.csv"
faq_corpus = DocumentCorpus(LocalFileLoader(str(faq_path), text_field="question")).load()   # step 1: core metadata + record.index
faq_core   = set(faq_corpus.docs[0].metadata)

sorted(faq_core)                                                                            # before enrich(): no source.table / record.fields yet

['content.metrics.length',
 'content.preview',
 'record.index',
 'source.info.relative',
 'source.info.uri',
 'source.properties.extension',
 'source.properties.is_remote',
 'timestamp.acquired_at']

In [12]:
faq_corpus.enrich()                                                    # step 2: source.table + record.fields, added in place

display(sorted(set(faq_corpus.docs[0].metadata) - faq_core))           # after enrich(): the keys it added
faq_corpus.df_docs[["page_content", "meta.record.fields.category", "meta.content.metrics.token_sz"]].head()

['record.fields.answer',
 'record.fields.category',
 'record.fields.id',
 'source.table.num_columns',
 'source.table.num_rows']

,page_content,meta.record.fields.category,meta.content.metrics.token_sz
0,How do I reset my password?,Account,7
1,What's your refund policy?,Billing,5
2,How do I cancel my subscription?,Billing,7
3,What payment methods do you accept?,Billing,7
4,How do I update my profile information?,Account,8


In [13]:
orders_path   = Path(read_root_dir()) / "data/cs/datasets/orders.csv"
orders_corpus = DocumentCorpus(LocalFileLoader(str(orders_path))).load()   # step 1: core metadata + record.index (no text)
orders_core   = set(orders_corpus.docs[0].metadata)

sorted(orders_core)                                                        # before enrich(): no source.table / record.fields yet

['content.metrics.length',
 'content.preview',
 'record.index',
 'source.info.relative',
 'source.info.uri',
 'source.properties.extension',
 'source.properties.is_remote',
 'timestamp.acquired_at']

In [14]:
orders_corpus.enrich()                                                 # step 2: source.table + record.fields, added in place

display(sorted(set(orders_corpus.docs[0].metadata) - orders_core))     # after enrich(): the keys it added
for ix, row in orders_corpus.df_metadata.filter(like='fields').iterrows():
    print(ix, row)

['record.fields.carrier',
 'record.fields.customer_email',
 'record.fields.customer_name',
 'record.fields.eta',
 'record.fields.item',
 'record.fields.order_date',
 'record.fields.order_id',
 'record.fields.payment_method',
 'record.fields.quantity',
 'record.fields.shipping_city',
 'record.fields.shipping_state',
 'record.fields.status',
 'record.fields.total_price',
 'record.fields.tracking_number',
 'record.fields.unit_price',
 'source.table.num_columns',
 'source.table.num_rows']

0 meta.record.fields.order_id                                      ORD-12345
meta.record.fields.customer_name                                Sarah Chen
meta.record.fields.customer_email                   sarah.chen@example.com
meta.record.fields.order_date                                   2025-12-01
meta.record.fields.status                                          shipped
meta.record.fields.carrier                                           FedEx
meta.record.fields.tracking_number                              1234567890
meta.record.fields.eta                                   December 11, 2025
meta.record.fields.item               Wireless Noise-Cancelling Headphones
meta.record.fields.quantity                                              1
meta.record.fields.unit_price                                       249.99
meta.record.fields.total_price                                      249.99
meta.record.fields.shipping_city                                    Austin
meta.record.fields.ship